# Matrices explicit calculation

In [ ]:
import sympy as sp
import numpy as np
import scipy as scp

def Matrices_Legendre(N):
    """Calculates A^{(k)}, when N_PC = N, in the case of Uniform distribution, for every k"""

    A = [[[0 for i in range(2*N+2)] for j in range(2*N+2)] for k in range(N+1)]
    
    if N>=0:
        for i in range(0,2*N+2):                    # It is necessary to calculate them up to an higher horder, since if i = N, the recurrence relation written below needs i+1 and so on
            A[0][i][i] = sp.Rational(2,2*i+1)

    if N>=1:
        for i in range(1,2*N+2):
            A[1][i-1][i] = sp.Rational(2*i,4*i**2-1)
            A[1][i][i-1] = sp.Rational(2*i,4*i**2-1)

    for k in range(2,N+1):
        for i in range (0, 2*(N+1)-k):
            for j in range(i, 2*(N+1)-k):
                if i==N:
                    temp = sp.Rational((2*k-1)*(i+1),k*(2*i+1))*A[k-1][i+1][j]+sp.Rational((2*k-1)*i,k*(2*i+1))*A[k-1][i-1][j]-sp.Rational(k-1,k)*A[k-2][i][j]
                else:
                    temp = sp.Rational((2*k-1)*(i+1),k*(2*i+1))*A[k-1][i+1][j]+sp.Rational((2*k-1)*i,k*(2*i+1))*A[k-1][i-1][j]-sp.Rational(k-1,k)*A[k-2][i][j]
                A[k][i][j] = temp
                A[k][j][i] = temp
    B = [[[A[k][i][j] for i in range(0,N+1)]for j in range(0,N+1)]for k in range(0,N+1)]
    for k in range(0,N+1):
        B[k]=(np.array(B[k],dtype=float))
    return B

In [ ]:
np.set_printoptions(precision=2, suppress=True)

N_PC = 5
A = Matrices_Legendre(N_PC)

A0_inv = np.linalg.inv(A[0])

# for k in range(0,N_PC+1):
#     eigenvalues = np.linalg.eigvals(A[k])
#     print(eigenvalues)

for i in range(1,N_PC+1):
    for j in range(1,N_PC+1):
        M = A0_inv@A[i]
        N = A0_inv@A[j]
        print(np.matrix(M@N-N@M))

In [6]:
def diagonal_system(M1,M2):
    """Given M1 sdp and M2 symmetric, finds a basis w.r.to the two matrices are both diagonal"""
    R = np.linalg.cholesky(M1)
    Rinv = np.linalg.inv(R)

    M1tilda = Rinv@M1@Rinv
    M2tilda = Rinv@M2@Rinv
    eigenvalues, eigenvectors = np.linalg.eig(M2tilda)
    U = eigenvectors
    P = Rinv@U
    I = P.T@M1@P
    Lambda = P.T@M2@P
    return P, I, Lambda, eigenvalues

N_PC = 5

c1, c2, c3, c4, c5, c6 = sp.symbols("c1 c2 c3 c4 c5 c6")

c = np.array([c1, c2, c3, c4, c5, c6])

A = Matrices_Legendre(N_PC)

P1, I1, Lambda1, eigenvalues1 = diagonal_system(A[0],A[1])
# print(P1,"\n",I1,"\n",Lambda1)

P2, I2, Lambda2, eigenvalues2 = diagonal_system(A[0],A[2])
# print(P2,"\n",I2,"\n",Lambda2)

B = np.linalg.inv(P2)*P1
print(B)
expr1 = c@B@c.T
expr2 = c*eigenvalues2
print(expr1.expand())
print((expr2@B@c.T).expand())


[[ 0.12114471  0.          0.16101573  0.         -0.08448806  0.        ]
 [-0.56005879 -0.          0.09819691  0.         -0.04160603  0.        ]
 [ 0.70691215  0.         -0.09382678  0.         -0.25790604  0.        ]
 [ 0.          0.20064472 -0.         -0.35126837 -0.         -0.12990334]
 [-0.          0.559057   -0.         -0.28550017  0.          0.21418968]
 [-0.          0.53938535 -0.         -0.03255204 -0.          0.29952115]]
0.121144710344655*c1**2 - 0.560058786652107*c1*c2 + 0.867927881551599*c1*c3 - 0.0844880573763988*c1*c5 + 0.0981969067118732*c2*c3 + 0.200644721945114*c2*c4 + 0.517450972942494*c2*c5 + 0.539385350250346*c2*c6 - 0.093826781367699*c3**2 - 0.257906043371711*c3*c5 - 0.351268366455412*c4**2 - 0.285500168690014*c4*c5 - 0.162455381909235*c4*c6 + 0.214189684981518*c5*c6 + 0.29952114751876*c6**2
0.0974305233410107*c1**2 + 0.232195515023765*c1*c2 + 0.239631389250896*c1*c3 - 0.0679494434616978*c1*c5 - 0.0407115857676335*c2*c3 - 0.0507499863987719*c2*c4 + 